# PySpark 13: performance tuning

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-13-performance-tuning.ipynb)

The diagnosis is in: a skewed join, too many small files, a shuffle that spills. Now you change one thing at a
time, measure, and prove the output did not change. Good tuning is evidence first and configuration last.

This lesson treats tuning as an experiment, then works through the levers in order of impact: shuffle partition
counts, skew and salting, join strategies, repartitioning, file sizes, partition pruning and dynamic pruning,
bucketing and statistics, Python UDF costs, streaming batch sizes and table layout. It ends with arithmetic
exercises, real incidents and an interview checklist.

Each section explains how something works, where it matters and what to watch out for, then runs small examples. Read each output, then the **What to notice** note under it. Run the cells in order.

**Prerequisites:** [PySpark 12: internals and monitoring](pyspark-12-internals-and-monitoring.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The small tables from
[PySpark 1](pyspark-01-how-spark-runs.ipynb#sample-data-used-in-every-module) are too small to show a shuffle or
a straggler, so this module builds a larger pair of tables instead.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date

from pyspark.sql import functions as F

import contextlib
import io


def plan_of(df, mode="formatted"):
    """Return the text `df.explain(mode)` prints, so a cell can pick out the lines that matter."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue()

`big_orders` has 200,000 rows, and customer `0` owns 40 percent of them: the "guest checkout" key that makes
real joins skew. `big_customers` is a 5,000-row dimension. The cell also resets the session to a static
baseline (16 shuffle partitions, AQE off), so each lever can be switched on deliberately.

In [ ]:
# Tuning lessons start from a static baseline; later cells switch levers on one at a time.
spark.conf.set("spark.sql.shuffle.partitions", "16")
spark.conf.set("spark.sql.adaptive.enabled", "false")

from pyspark.sql.functions import spark_partition_id


def partition_sizes(df):
    """Row count per partition, in partition order: the skew signal the Stages tab shows."""
    return [r["count"] for r in df.groupBy(spark_partition_id().alias("p")).count().orderBy("p").collect()]

ROWS, CUSTOMERS, HOT_SHARE = 200_000, 5_000, 0.4
big_orders = spark.range(ROWS).select(
    F.col("id").alias("order_id"),
    F.when(F.col("id") % 10 < HOT_SHARE * 10, F.lit(0)).otherwise(F.col("id") % CUSTOMERS).alias("customer_id"),
    F.date_add(F.lit("2026-10-01").cast("date"), (F.col("id") % 30).cast("int")).alias("order_date"),
    (F.col("id") % 997 / 10).cast("decimal(12,2)").alias("amount"),
    F.element_at(F.array(F.lit("paid"), F.lit("shipped"), F.lit("placed")), (F.col("id") % 3 + 1).cast("int")).alias("status"),
)
big_customers = spark.range(CUSTOMERS).select(
    F.col("id").alias("customer_id"),
    F.concat(F.lit("segment-"), (F.col("id") % 4).cast("string")).alias("customer_segment"),
)

# A partitioned Parquet copy, so scans, pruning and pushdown behave like a real table
ORDERS_PATH = os.path.join(WORK_DIR, "big_orders")
big_orders.write.mode("overwrite").partitionBy("order_date").parquet(ORDERS_PATH)
big_orders = spark.read.parquet(ORDERS_PATH)

hot = big_orders.where("customer_id = 0").count()
print(f"rows: {big_orders.count():,}  hot-key rows: {hot:,}")

## Treat tuning as a measured experiment

Start with evidence, not configuration: reproduce the slow job on representative data, read the UI and the
plan, name the bottleneck (scan, shuffle, skew, spill, join, small files, driver memory or the sink), change
one thing, and measure again on the same data, code and cluster.

A stage finishes only when its slowest task does. A stage with 1,000 balanced tasks and one straggler is as
slow as the straggler, however much memory you add. Record the same fields before and after: wall clock,
shuffle read and write, spill, task count, maximum task duration, GC time, output file count and cost per
run. Shuffle bytes are the leading indicator; wall clock is what the business sees but also the noisiest, so
compare medians over several runs.

**Where it matters**
- **Proving a change:** "shuffle fell from 1.2 TB to 20 GB and spill to zero" is evidence; "I enabled AQE" is not.
- **Cost reviews:** halving runtime on a cluster twice the size buys latency, not efficiency.

**Watch out for**
- Benchmarking with `count()`. The optimizer can skip columns and work that your real write needs; measure the action you want to improve.
- Comparing runs with different cache states, or changing two settings at once.
- Forgetting to check that the output is identical after the change.

### Example

Compare a baseline and a tuned run field by field.

In [ ]:
import json

# Record the same fields before and after, then compare like with like.
def summarize(before, after):
    def delta(key):
        b, a = before[key], after[key]
        change = (a - b) / b * 100 if b else 0.0
        return f"{b:,} -> {a:,} ({change:+.1f}%)"

    for key in ("wall_clock_s", "shuffle_read_bytes", "shuffle_write_bytes",
                "disk_bytes_spilled", "max_task_s", "task_count"):
        print(f"{key:22s} {delta(key)}")

baseline = {"wall_clock_s": 3600, "shuffle_read_bytes": 1_200_000_000_000,
            "shuffle_write_bytes": 1_200_000_000_000, "disk_bytes_spilled": 8_000_000_000,
            "max_task_s": 540, "task_count": 200_000}
tuned = {"wall_clock_s": 1_200, "shuffle_read_bytes": 20_000_000_000,
         "shuffle_write_bytes": 20_000_000_000, "disk_bytes_spilled": 0,
         "max_task_s": 95, "task_count": 40_000}

summarize(baseline, tuned)
print(json.dumps({"shuffle_reduction_pct": round(
    (1 - tuned["shuffle_read_bytes"] / baseline["shuffle_read_bytes"]) * 100, 1)}))

## Choose the number of shuffle partitions

`spark.sql.shuffle.partitions` (default 200) sets the reduce fan-out of every shuffle. Too few partitions
give large tasks that spill; too many give scheduling overhead and tiny output files. Derive it from a target
size of roughly 100 to 200 MB per partition: `partitions = ceil(shuffle_bytes / target_size)`.

With AQE on, the setting is only the starting count: `spark.sql.adaptive.advisoryPartitionSizeInBytes`
(64 MB by default) is the size AQE coalesces toward once it sees the real map output.

**Where it matters**
- **A 2 TB shuffle:** about 16,000 partitions at 128 MB, or 32,000 at 64 MB.
- **Tiny data in tests:** a handful of partitions instead of 200 empty tasks.

**Watch out for**
- A huge round number "just in case" with AQE off: tens of thousands of tiny tasks and files.
- Changing the setting after building a DataFrame applies to the next action, not to plans already executed.

### Example

With AQE off the aggregation keeps the configured 16 partitions; with AQE on it coalesces them. Then compute counts from bytes.

In [ ]:
by_customer = lambda: big_orders.groupBy("customer_id").agg(F.sum("amount").alias("revenue"))  # noqa: E731

spark.conf.set("spark.sql.adaptive.enabled", "false")
static_partitions = by_customer().rdd.getNumPartitions()

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
adaptive_partitions = by_customer().rdd.getNumPartitions()

print("AQE off:", static_partitions, "partitions | AQE on:", adaptive_partitions)

In [ ]:
import math


def shuffle_partitions_for(shuffle_bytes, target_bytes=128 * 1024**2):
    return math.ceil(shuffle_bytes / target_bytes)

for label, size in [("200 GB", 200 * 1024**3), ("2 TB", 2 * 1024**4)]:
    print(f"{label:>6} shuffle -> {shuffle_partitions_for(size):,} partitions at 128 MB")

## Fix data skew with salting

When a few keys hold most rows, one task owns most of the work. **Salting** adds a random bucket to the hot
side and replicates the small side across all buckets, so the hot key spreads over many tasks. Aggregations
salt in two phases: aggregate by `(key, salt)`, then by `key`. Use it when broadcasting is impossible and
AQE skew handling is not enough, and only after a key count confirmed the skew.

**Where it matters**
- **Guest checkouts, null or `UNKNOWN` keys,** and a few very large tenants.
- **Joins where both sides are too large to broadcast.**

**Watch out for**
- Salting without confirmed skew adds a shuffle for nothing.
- The small side grows by the salt factor: a 20 MB dimension salted 32 ways is 640 MB.
- A salt computed from a hash of the key alone does not help: every row of the hot key gets the same salt. Often the real fix is the source of the hot key (a null or placeholder).

### Example

Salt the join 16 ways and compare the largest partition; then salt an aggregation in two phases and check the totals match.

In [ ]:
salt_buckets = 16

orders_salted = big_orders.withColumn("salt", (F.rand(seed=42) * salt_buckets).cast("int"))
customers_salted = big_customers.crossJoin(spark.range(salt_buckets).withColumnRenamed("id", "salt"))

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # force a shuffle join for the demo
plain = big_orders.join(big_customers, "customer_id")
salted = orders_salted.join(customers_salted, ["customer_id", "salt"]).drop("salt")

hot_before = max(partition_sizes(big_orders.repartition(16, "customer_id")))
hot_after = max(partition_sizes(orders_salted.repartition(16, "customer_id", "salt")))
print(f"largest join partition: {hot_before:,} rows before salting, {hot_after:,} after")

spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

In [ ]:
partial = orders_salted.groupBy("customer_id", "salt").agg(F.sum("amount").alias("partial_amount"))
final = partial.groupBy("customer_id").agg(F.sum("partial_amount").alias("total_amount"))

direct = big_orders.groupBy("customer_id").agg(F.sum("amount").alias("total_amount"))
print("two-phase salted aggregate matches the direct aggregate")

## Tune joins and broadcasts

Start with the size and distribution of both sides: broadcast a small side, filter and select before joining,
deduplicate dimensions so facts are not multiplied, and handle hot keys.

- `BroadcastHashJoin` ships the small side to every executor; the large side never shuffles.
- `SortMergeJoin` shuffles and sorts both sides; it scales to two large inputs and spills gracefully.
- `ShuffledHashJoin` shuffles both sides but builds a hash table instead of sorting (chosen by AQE, or with
  `spark.sql.join.preferSortMergeJoin=false`).
- Non-equi joins fall back to `BroadcastNestedLoopJoin` or a Cartesian product: almost always a mistake.

Spark broadcasts automatically below `spark.sql.autoBroadcastJoinThreshold` (10 MB; `-1` disables it). At
runtime the small side is collected to the driver and then sent to executors, so the costs are driver memory,
executor memory and broadcast latency.

**Where it matters**
- **Fact-to-dimension joins,** where a broadcast removes the shuffle entirely.
- **A broadcast that stopped broadcasting** because a dimension grew past the threshold.

**Watch out for**
- Joining on an expression such as `lower(a.id) == lower(b.id)` instead of a clean key column.
- A dimension that is not unique on the key silently multiplies facts.
- Raising the threshold globally to make one join broadcast. Hint that join, after checking its size.

### Example

A non-unique dimension inflates the join; then the plan switches from sort-merge to broadcast when the small side is hinted.

In [ ]:
dim_with_history = big_customers.unionByName(big_customers.where("customer_id < 10"))  # two versions of 10 rows

fanned_out = big_orders.join(dim_with_history, "customer_id").count()
deduped = big_orders.join(dim_with_history.dropDuplicates(["customer_id"]), "customer_id").count()
print(f"rows after join: {fanned_out:,} with a non-unique dimension, {deduped:,} after dedupe")

In [ ]:
from pyspark.sql.functions import broadcast

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
shuffled_plan = plan_of(big_orders.join(big_customers, "customer_id"))
hinted_plan = plan_of(big_orders.join(broadcast(big_customers), "customer_id"))
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

print("threshold off -> SortMergeJoin; broadcast() hint -> BroadcastHashJoin")

## Repartition, coalesce and partition by column

`repartition(n)` is a full shuffle (round-robin, or hash when you pass columns); it can raise or lower the
count and balances by construction. `coalesce(n)` merges existing partitions without a shuffle, can only lower
the count, and keeps any imbalance because it never splits a partition.

`repartition(n, col)` hash-partitions so equal keys always meet, which is what a keyed join, aggregation or
dedup needs. Before a partitioned write, `repartition("order_date")` followed by `partitionBy("order_date")`
writes about one file per date; without it, each task writes one file per date it holds.

**Where it matters**
- **After a selective filter:** 1,000 sparse partitions down to a sensible count.
- **Before a partitioned write,** to control the number of files per partition.

**Watch out for**
- `coalesce(1)` on a large result funnels the whole write onto one core.
- All nulls hash to one partition, and one hot value makes one partition huge whatever `n` is.
- Repartitioning by one key and then shuffling on another: two exchanges where one was needed.

### Example

Compare partition sizes after hashing on a skewed key, coalescing and repartitioning; then count files written with and without `repartition` by date.

In [ ]:
skewed = big_orders.repartition(8, "customer_id")  # the hot key lands in a single partition
merged = skewed.coalesce(4)  # narrow: merges whole partitions, keeps the hot one intact
balanced = skewed.repartition(4)  # full round-robin shuffle

for name, frame in [("hash by customer", skewed), ("coalesce(4)", merged), ("repartition(4)", balanced)]:
    print(f"{name:<17} {partition_sizes(frame)}")

exchanges = lambda df: plan_of(df, "simple").count("Exchange ")  # noqa: E731

In [ ]:
def file_count(path):
    return sum(name.endswith(".parquet") for _, _, names in os.walk(path) for name in names)

scattered_path = os.path.join(WORK_DIR, "scattered")
clustered_path = os.path.join(WORK_DIR, "clustered")
big_orders.repartition(8).write.mode("overwrite").partitionBy("order_date").parquet(scattered_path)
big_orders.repartition("order_date").write.mode("overwrite").partitionBy("order_date").parquet(clustered_path)

print("files without repartition by date:", file_count(scattered_path))
print("files with repartition by date   :", file_count(clustered_path))

## Avoid small files

Spark packs files into scan tasks using `spark.sql.files.maxPartitionBytes` (128 MB) and
`spark.sql.files.openCostInBytes` (4 MB, the modelled cost of opening a file). With many tiny files, listing,
opening and footer reads dominate, and table formats carry more per-file metadata. Control the file count
when writing, cap rows per file with `maxRecordsPerFile`, and compact what already exists with the table
format:

```sql
-- Delta
OPTIMIZE main.silver.orders WHERE order_date >= date '2026-10-01';
-- Iceberg (Spark procedure)
CALL catalog.system.rewrite_data_files(table => 'silver.orders');
```

**Where it matters**
- **Streaming sinks** that write a few files every minute.
- **Over-partitioned tables:** 500,000 files of 1 MB compacted to 1 GB files leaves about 500.

**Watch out for**
- Rewriting a whole table to fix one partition.
- Leaving write-time compaction (`delta.autoOptimize.optimizeWrite`, `autoCompact`) off on streaming tables.

### Example

Write one day as 24 files, then compact it into one (fine here only because the day is tiny).

In [ ]:
small_path = os.path.join(WORK_DIR, "small_files")
compact_path = os.path.join(WORK_DIR, "compacted")
day = big_orders.where("order_date = date '2026-10-05'")

day.repartition(24).write.mode("overwrite").parquet(small_path)
spark.read.parquet(small_path).coalesce(1).write.mode("overwrite").parquet(compact_path)

print("files before compaction:", file_count(small_path), "| after:", file_count(compact_path))

## Prune partitions and push down predicates

**Partition pruning** uses the directory layout: Spark evaluates the predicate on partition values and lists
the survivors under `PartitionFilters`. **Pushdown and data skipping** work inside files: Parquet row-group
min/max values and Delta per-file statistics let the reader skip data, shown under `PushedFilters`. Wrapping a
regular column in a function hides the literal from the reader.

**Dynamic partition pruning** (DPP) applies the same idea at runtime: when a partitioned fact table is joined
to a filtered dimension, Spark first computes the dimension's matching keys and uses them to prune the
fact table's partitions.

Catalyst pushes deterministic filters and projections toward the scan, but not through aggregations,
windows, `distinct`, Python UDFs or non-deterministic expressions, so write the filter early anyway.

**Where it matters**
- **Date-partitioned facts:** a one-day query over five years reads about 1/1825 of the files.
- **Star-schema queries** that filter a calendar or store dimension and join it to a partitioned fact.

**Watch out for**
- `lower(country) = 'us'`, `cast(ts as string) like '2026-10%'` or comparing a timestamp with a string literal defeat pushdown on a regular column.
- Delta collects statistics for the first 32 columns by default (`delta.dataSkippingNumIndexedCols`); put filter columns early.

### Example

A raw predicate is pushed down; a wrapped one is not; an expression on the partition column still prunes. Then Catalyst moves a late filter down to the same scan.

In [ ]:
def plan_line(df, prefix):
    return next(line.strip() for line in plan_of(df).splitlines() if line.strip().startswith(prefix))

raw_predicate = big_orders.where("status = 'paid'")
wrapped_predicate = big_orders.where("lower(status) = 'paid'")
partition_expression = big_orders.where("date_format(order_date, 'yyyy-MM-dd') = '2026-10-05'")

print(plan_line(raw_predicate, "PushedFilters"))
print(plan_line(wrapped_predicate, "PushedFilters"))
print(plan_line(partition_expression, "PartitionFilters"))

In [ ]:
early = (
    big_orders.where("order_date >= date '2026-10-20'")
    .select("customer_id", "amount")
    .join(big_customers, "customer_id")
    .groupBy("customer_segment").agg(F.sum("amount").alias("revenue"))
)
late = (
    big_orders.join(big_customers, "customer_id")
    .where("order_date >= date '2026-10-20'")
    .groupBy("customer_segment").agg(F.sum("amount").alias("revenue"))
)
print("Catalyst pushed the late filter down to the same scan:", plan_line(late, "PartitionFilters"))

### Example: dynamic partition pruning

Join the date-partitioned orders to a calendar filtered to one holiday. The plan prunes order partitions with a dynamic pruning expression.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
calendar = spark.createDataFrame(
    [(date(2026, 10, 5), "holiday"), (date(2026, 10, 6), "normal")], "order_date date, day_type string"
)
holiday_orders = big_orders.join(calendar.where("day_type = 'holiday'"), "order_date")

print(plan_line(holiday_orders, "PartitionFilters"))
print("orders on the holiday:", holiday_orders.count(), "of", big_orders.count())

**What to notice:** The order scan's `PartitionFilters` contains a `dynamicpruningexpression`: at runtime Spark computes which
`order_date` values survive the calendar filter and reads only those partitions, one day of thirty. The
query never names a date on `big_orders` itself.

## Bucket tables and keep statistics current

**Bucketing** (`bucketBy(n, col)` with `saveAsTable`) hash-partitions a table into a fixed number of files
by a key when it is written. Two tables bucketed the same way on the join key can be joined without a
shuffle, because matching keys are already in matching buckets. It pays off for large tables joined on the
same key over and over.

The planner chooses join strategies from **statistics**. `ANALYZE TABLE ... COMPUTE STATISTICS` records row
counts and sizes (and, `FOR COLUMNS`, distinct counts and min/max values) in the catalog; the cost-based
optimizer (`spark.sql.cbo.enabled`) uses column statistics to order joins and estimate filters.

**Where it matters**
- **Repeated large joins** on the same key, such as orders to order lines by `order_id`.
- **Wrong join choices** caused by missing or stale statistics on catalog tables.

**Watch out for**
- Bucketing only helps when both sides use the same key and compatible bucket counts, and readers go through the catalog table.
- Each write task can create one file per bucket, so bucketing large writes multiplies file counts.
- Delta and Iceberg prefer clustering (liquid clustering, Z-order, sort orders) over Hive-style buckets.
- Statistics go stale after large loads; refresh them as part of the load.

### Example

Bucket both tables by `customer_id` into 8 buckets. With broadcasting off, their join plans no exchange at
all. Then compute statistics and read them back from the catalog.

In [ ]:
big_orders.write.bucketBy(8, "customer_id").sortBy("customer_id").mode("overwrite").saveAsTable("orders_bucketed")
big_customers.write.bucketBy(8, "customer_id").sortBy("customer_id").mode("overwrite").saveAsTable("customers_bucketed")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
bucketed_join = spark.table("orders_bucketed").join(spark.table("customers_bucketed"), "customer_id")
plain_join = big_orders.join(big_customers, "customer_id")
bucketed_exchanges = plan_of(bucketed_join, "simple").count("Exchange ")
plain_exchanges = plan_of(plain_join, "simple").count("Exchange ")
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")
print("exchanges - bucketed:", bucketed_exchanges, "| plain:", plain_exchanges)

spark.sql("ANALYZE TABLE customers_bucketed COMPUTE STATISTICS FOR COLUMNS customer_id")
stats = spark.sql("DESCRIBE EXTENDED customers_bucketed").where("col_name = 'Statistics'").first()
print(stats.data_type)

**What to notice:** With broadcasting off, the plain join shuffles both sides (two exchanges). The bucketed tables already
have matching keys in matching buckets, so their join plans no exchange at all. `ANALYZE TABLE` stored the
row count and size in the catalog, where the planner reads them.

## Tune Python UDFs and pandas UDFs

A row-at-a-time Python UDF serializes every row to a Python worker and back, is opaque to Catalyst, and breaks
whole-stage code generation. Order of preference: built-in expression, SQL or higher-order function,
vectorized pandas UDF, row UDF. Pandas UDFs amortize the boundary with Arrow batches
(`spark.sql.execution.arrow.maxRecordsPerBatch`, 10,000 rows by default); `mapInPandas` can change the row
count and `applyInPandas` runs once per group.

**Where it matters**
- **Hot paths** that call a UDF on every row of a large table.
- **ML scoring,** where a pandas UDF over a model is the right tool.

**Watch out for**
- On Databricks, Photon accelerates native operators but not Python UDFs, so a UDF in the hot path loses the speedup.
- A pandas UDF still crosses the JVM boundary; filter and aggregate in the JVM first.

### Example

The same tier rule as a built-in and as a UDF: same answer, but only the UDF plan leaves the JVM.

In [ ]:
from pyspark.sql import types as T


@F.udf(T.StringType())
def tier_udf(amount):
    return "high" if amount is not None and amount >= 90 else "standard"

builtin_tier = big_orders.withColumn("tier", F.when(F.col("amount") >= 90, "high").otherwise("standard"))
udf_tier = big_orders.withColumn("tier", tier_udf("amount"))

print("same result; only the UDF plan leaves the JVM")

## Tune Structured Streaming and table layout

A streaming query is a micro-batch loop: each trigger runs a normal job over newly available data. Bound each
batch with `maxOffsetsPerTrigger` (Kafka) or `maxFilesPerTrigger` (files), choose a trigger interval each batch
can finish within, bound state with watermarks, and keep one stable checkpoint per stream.

Layout changes the cost of every future query. Aim for 128 MB to 1 GB files; `OPTIMIZE` compacts small files,
`ZORDER BY (col)` co-locates values for data skipping, and **liquid clustering** (`CLUSTER BY (cols)`) records
clustering keys in metadata and re-clusters incrementally. Liquid clustering cannot be combined with
`partitionBy` or `ZORDER`. `VACUUM` removes unreferenced files older than the retention (7 days by default);
it saves storage, not time, and ends time travel for that window.

```sql
OPTIMIZE main.silver.orders WHERE order_date >= date '2026-10-01' ZORDER BY (customer_id);
ALTER TABLE main.silver.orders CLUSTER BY (order_date, customer_id);
VACUUM main.silver.orders RETAIN 168 HOURS;
```

**Where it matters**
- **Streams whose latency keeps climbing** as the source grows.
- **Tables read by many queries,** where layout work pays back on every read.

**Watch out for**
- Stateful dedup without a watermark: state grows until the job slows down a week later.
- A full-table scan inside `foreachBatch` that repeats every micro-batch.
- Optimizing on a schedule instead of on evidence, and partitioning by a high-cardinality column.

### Example

With a 60-second trigger, batches start taking 90 seconds as a topic grows, so each batch starts later with
more data and latency climbs without bound. Capping `maxOffsetsPerTrigger` and widening the trigger to two
minutes stabilizes it:

```python
(
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker1:9092")
    .option("subscribe", "orders")
    .option("maxOffsetsPerTrigger", 500_000)
    .load()
    .writeStream
    .option("checkpointLocation", "s3://acme-checkpoints/orders/")
    .trigger(processingTime="2 minutes")
    .toTable("main.silver.orders")
)
```

## Tuning exercises

Standard-library models of the quantities you read from the UI and `explain`: shuffle bytes, broadcast
thresholds, partition row counts and executor arithmetic. Work out each answer before running the cell; the
asserts check the formula, not a real cluster.

### Try it 1. Choose shuffle partitions

**Task.** A shuffle moves 640 GB (binary units). Compute the number of shuffle partitions needed for a 128 MB target partition size, then for a 256 MB target.

**Hint.** `partitions = ceil(shuffle_bytes / target_bytes)`. One GB here is `1024` MB.

**Expected result.** `5,120` partitions for 128 MB and `2,560` for 256 MB.

In [ ]:
import math


def shuffle_partitions(shuffle_gb, target_mb):
    shuffle_mb = shuffle_gb * 1024
    return math.ceil(shuffle_mb / target_mb)

partitions_128 = shuffle_partitions(640, 128)
partitions_256 = shuffle_partitions(640, 256)
print(f"640 GB at 128 MB/partition -> {partitions_128:,} partitions")
print(f"640 GB at 256 MB/partition -> {partitions_256:,} partitions")

### Try it 2. Broadcast threshold decision

**Task.** Using the default auto-broadcast threshold of 10 MB, decide whether a 7.5 MB dimension and a 45 MB dimension would be broadcast. Then show the driver and network cost of forcing the 45 MB dimension to broadcast across 60 executors.

**Hint.** Compare bytes against `10 * 1024 * 1024`. A forced broadcast replicates the relation to every executor.

**Expected result.** The 7.5 MB dimension broadcasts, the 45 MB one does not, and forcing it moves about `2,700` MB in total.

In [ ]:
MB = 1024 * 1024
THRESHOLD = 10 * MB
EXECUTORS = 60


def would_broadcast(size_mb):
    return size_mb * MB <= THRESHOLD

small, large = 7.5, 45.0
print(f"{small} MB -> broadcast: {would_broadcast(small)}")
print(f"{large} MB -> broadcast: {would_broadcast(large)}")

network_mb = large * EXECUTORS
print(f"forced broadcast network = {network_mb:,.0f} MB")

### Try it 3. Detect skew from per-partition row counts

**Task.** Given the row counts of eight shuffle partitions, compute the median, the maximum, and the ratio of max to median. Flag the stage as skewed when the max exceeds five times the median.

**Hint.** Sort the counts and take the middle value, averaging the two middle values for an even count.

**Expected result.** Median `101`, maximum `5,000`, ratio about `49.5`, and the stage is skewed.

In [ ]:
counts = [100, 105, 98, 102, 100, 5000, 99, 103]


def median(values):
    ordered = sorted(values)
    mid = len(ordered) // 2
    if len(ordered) % 2 == 1:
        return ordered[mid]
    return (ordered[mid - 1] + ordered[mid]) / 2

med = median(counts)
peak = max(counts)
ratio = peak / med
skewed = ratio > 5

print(f"median = {med:.1f}, max = {peak}, ratio = {ratio:.1f}")
print("skewed:", skewed)

### Try it 4. Salt a hot key and recompute the distribution

**Task.** One million rows are keyed by customer. The key `guest` holds 40 percent and four normal keys split the rest evenly. Compute the largest partition share before salting, then salt `guest` into 32 buckets evenly and recompute it.

**Hint.** After salting, the largest partition is either a salted bucket of the hot key or one of the normal keys, whichever is larger.

**Expected result.** The largest share falls from `0.40` to `0.15`, roughly a `2.67x` improvement.

In [ ]:
TOTAL = 1_000_000
SALT_BUCKETS = 32
hot = int(TOTAL * 0.40)
normal = (TOTAL - hot) / 4

before = hot / TOTAL
salted_bucket = hot / SALT_BUCKETS
after = max(salted_bucket, normal) / TOTAL

print(f"before: hot key share = {before:.2f}")
print(f"after : largest share = {after:.2f} (salted bucket = {salted_bucket:,.0f} rows)")
print(f"improvement = {before / after:.2f}x")

### Try it 5. Decide repartition versus coalesce

**Task.** After a filter, a DataFrame has 1,000 partitions holding 4 GB in total and the downstream step wants 16 tasks. Decide between `coalesce(16)` and `repartition(16)` using a 128 MB target partition size. Repeat for a 512 MB result with 4 target tasks.

**Hint.** `coalesce` merges whole upstream partitions without a shuffle, so it is safe only when the merged partition fits the target. Otherwise you need `repartition`.

**Expected result.** 4 GB into 16 tasks is 256 MB each, so use `repartition`; 512 MB into 4 tasks is 128 MB each, so `coalesce` is fine.

In [ ]:
MB = 1024 * 1024


def choose_op(total_mb, target_tasks, target_mb=128):
    per_task = total_mb / target_tasks
    choice = "coalesce" if per_task <= target_mb else "repartition"
    return choice, per_task

choice_a, per_a = choose_op(4 * 1024, 16)
choice_b, per_b = choose_op(512, 4)
print(f"4096 MB / 16 = {per_a:.0f} MB per task -> {choice_a}")
print(f" 512 MB /  4 = {per_b:.0f} MB per task -> {choice_b}")

### Try it 6. Estimate executor count from cores and memory

**Task.** A cluster has 10 nodes with 32 cores and 256 GB of RAM each. Leave one core and 1 GB per node for the operating system and the NodeManager. With 5 cores per executor, compute executors per node, total executors, total cores used, and the memory per executor.

**Hint.** `executors_per_node = floor((cores_per_node - 1) / cores_per_executor)`, then divide the usable memory by that count.

**Expected result.** `6` executors per node, `60` in total, `30` cores used per node, and about `42` GB of memory each.

In [ ]:
NODES = 10
CORES_PER_NODE = 32
MEM_PER_NODE_GB = 256
CORES_PER_EXECUTOR = 5

usable_cores = CORES_PER_NODE - 1
usable_mem = MEM_PER_NODE_GB - 1

executors_per_node = usable_cores // CORES_PER_EXECUTOR
cores_used = executors_per_node * CORES_PER_EXECUTOR
mem_per_executor = usable_mem // executors_per_node
total_executors = executors_per_node * NODES

print(f"executors per node = {executors_per_node}")
print(f"total executors    = {total_executors}")
print(f"cores used per node= {cores_used} of {CORES_PER_NODE}")
print(f"memory per executor= {mem_per_executor} GB")

### Try it 7. Compute AQE coalescing of small partitions

**Task.** A shuffle produces 200 partitions of 2 MB each. AQE coalesces adjacent partitions toward `spark.sql.adaptive.advisoryPartitionSizeInBytes = 64m`. Compute the number of partitions after coalescing and the resulting average partition size.

**Hint.** `coalesced = ceil(total_bytes / advisory_bytes)`. Each coalesced partition packs whole input partitions, so the total is unchanged.

**Expected result.** `400` MB coalesces into `7` partitions of about `57.1` MB each.

In [ ]:
import math

INPUT_PARTITIONS = 200
PARTITION_MB = 2
ADVISORY_MB = 64

total_mb = INPUT_PARTITIONS * PARTITION_MB
coalesced = math.ceil(total_mb / ADVISORY_MB)
avg_mb = total_mb / coalesced

print(f"total = {total_mb} MB across {INPUT_PARTITIONS} partitions")
print(f"coalesced -> {coalesced} partitions, average {avg_mb:.1f} MB")

### Try it 8. Compare two plans by estimated shuffle bytes

**Task.** Plan A is a sort-merge join that shuffles 1.2 TB in each direction. Plan B broadcasts a 300 MB dimension to 60 executors and shuffles the fact side not at all. Compare total bytes moved across the network and compute the reduction factor.

**Hint.** Plan A moves its shuffle write plus its shuffle read. Plan B moves the broadcast size once per executor. Convert everything to MB before comparing.

**Expected result.** Plan A moves `2.4` TiB, Plan B about `18,000` MB, a reduction of roughly `139.8x`.

In [ ]:
MB_PER_GIB = 1024

plan_a_write_mb = 1.2 * 1024 * MB_PER_GIB
plan_a_read_mb = 1.2 * 1024 * MB_PER_GIB
plan_a_total = plan_a_write_mb + plan_a_read_mb

broadcast_mb = 300
executors = 60
plan_b_total = broadcast_mb * executors

reduction = plan_a_total / plan_b_total
print(f"plan A moves {plan_a_total:,.0f} MB")
print(f"plan B moves {plan_b_total:,.0f} MB")
print(f"reduction = {reduction:.1f}x")

## Real-world tuning cases

### 1. A nightly job that regressed from 20 minutes to 3 hours

One tenant's data grew; stage 7 showed one task at 90 percent of the stage time plus spill -
skew plus a fixed, oversized partition count. Fix: enable AQE coalescing and skew join with a
128 MB advisory size. Runtime returned to minutes with no memory change.

### 2. A join on a hot customer key

One customer produced 40 percent of rows and AQE skew handling was not enough, so the join was
salted into 16 buckets (the salting section above). Measure the key distribution first - salting
adds a shuffle.

### 3. Small files after streaming ingestion

A streaming sink wrote thousands of tiny files per hour, and planning and file opens dominated
scans. Fix: compact with the table format (`OPTIMIZE`, `rewrite_data_files`) and cap output with
`spark.sql.files.maxRecordsPerFile` or a repartition before writing.

### 4. A broadcast join that stopped broadcasting

A "small" product dimension grew past 10 MB, so every run shuffled both sides. Fix: a deliberate
`broadcast()` hint or a higher threshold, after checking driver and executor memory.

### 5. Streaming latency that kept climbing

Each micro-batch consumed everything available and grew larger. Fix: `maxOffsetsPerTrigger` plus
a 2-minute trigger, and a watermark on stateful operators.

### 6. Driver OOM from collecting results

`collect()` and `toPandas()` on large frames killed the driver. Fix: aggregate in the cluster and
`limit()` any sample that must cross to the driver.

In every case the UI or the plan named the bottleneck before anyone changed a setting, and the
proof was a byte count or a task duration. In an interview give the numbers and the mechanism:
"stage 7 had one task at 90 percent of the stage time and spill; AQE split the skewed partition,
spill went to zero, and runtime fell from three hours to twenty minutes" - not "I enabled AQE".

## What tuning checklist should you use in interviews?

+ Inspect the Spark UI and the physical plan first.
+ Check input size, file count, and partition pruning.
+ Filter rows and select columns early; avoid Python UDFs.
+ Broadcast small dimensions, deduplicate dimensions, handle verified skew (AQE, then salting).
+ Size shuffle partitions and enable AQE.
+ `repartition` to balance, `coalesce` to reduce output partitions.
+ Cache only reused, expensive frames; never `collect()` large data.
+ Compact small files and write sensible file sizes.
+ Validate with runtime, shuffle and spill metrics, and output correctness.

Strong answer: "I would not start with executor memory. I would read the plan and the UI, decide
whether the bottleneck is scan, shuffle, skew, join, spill, or output files, apply the smallest
targeted fix, and measure again." Naming a config is much weaker than naming the evidence that
justifies it.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** How do you choose `spark.sql.shuffle.partitions`?

**A.** From bytes: `ceil(shuffle_bytes / target)` with a target of about 100 to 200 MB. With AQE on, set a generous starting count and let `advisoryPartitionSizeInBytes` coalesce.

**Q2.** When do you salt a join, and what does it cost?

**A.** When one key holds a large share of rows, broadcasting is impossible and AQE skew handling is not enough. Salting adds a shuffle and replicates the small side by the salt factor.

**Q3.** What is the difference between `repartition` and `coalesce`?

**A.** `repartition` is a full shuffle that balances partitions and can raise or lower the count; `coalesce` merges partitions without a shuffle, can only lower the count, and keeps any imbalance.

**Q4.** When does dynamic partition pruning apply?

**A.** When a partitioned table is joined on its partition column to a filtered side; Spark computes the matching keys at runtime and skips the other partitions.

**Q5.** Why add executor memory last?

**A.** Spill usually comes from partitions larger than per-task memory. More partitions, earlier filtering or a broadcast remove it for free; memory is the expensive lever.

**Q6.** What makes a strong answer to "how would you tune this job?"

**A.** Naming the evidence (the slow stage, its skew or spill, the plan's exchanges), the one targeted change, and the measurement that proved it, with the output unchanged.

## Key takeaways

- Measure before and after on the same data, change one thing, and check the output is identical.
- Size shuffle partitions from bytes; let AQE coalesce and split.
- One slow task out of hundreds is skew: AQE skew join first, then salt or fix the hot key.
- Broadcast only what is genuinely small, and confirm it in the plan.
- Write 128 MB to 1 GB files, prune partitions, and keep statistics current.

**Next:** Continue with [PySpark 14: production PySpark](pyspark-14-production-pyspark.ipynb).